In [57]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [58]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,3,15619304,Onio,502,France,Female,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,4,15701354,Boni,699,France,Female,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [59]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10002 entries, 0 to 10001
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowNumber        10002 non-null  int64  
 1   CustomerId       10002 non-null  int64  
 2   Surname          10002 non-null  str    
 3   CreditScore      10002 non-null  int64  
 4   Geography        10001 non-null  str    
 5   Gender           10002 non-null  str    
 6   Age              10001 non-null  float64
 7   Tenure           10002 non-null  int64  
 8   Balance          10002 non-null  float64
 9   NumOfProducts    10002 non-null  int64  
 10  HasCrCard        10001 non-null  float64
 11  IsActiveMember   10001 non-null  float64
 12  EstimatedSalary  10002 non-null  float64
 13  Exited           10002 non-null  int64  
dtypes: float64(5), int64(6), str(3)
memory usage: 1.2 MB


### Preprocessing 
##### 1. Removing irrelevant features
##### 2. Converting Gender column to numerical using Label encoding
##### 3. Removing the null values
##### 4. Converting geography column data to one-hot-encoding
##### 5. Scaling all the values



In [60]:
# Drop irrelevant features or columns
data=data.drop(['RowNumber', 'CustomerId','Surname'], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,608,Spain,Female,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,502,France,Female,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,699,France,Female,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,850,Spain,Female,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [61]:
# Convert gender column data to numbers using Label Encoder
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,608,Spain,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,502,France,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,699,France,0,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,850,Spain,0,43.0,2,125510.82,1,NaN,1.0,79084.10,0


In [62]:
# To fill the null values
data['Geography'].unique()

<ArrowStringArray>
['France', 'Spain', nan, 'Germany']
Length: 4, dtype: str

In [63]:
# Fill the null value of Geography column
data['Geography'].value_counts()
data['Geography']=data['Geography'].fillna('France')
data['Geography'].value_counts()

Geography
France     5015
Germany    2510
Spain      2477
Name: count, dtype: int64

In [64]:
# Fill the null value of age column
data['Age']=data['Age'].fillna(data['Age'].mean())
data['Age']

0        42.0
1        41.0
2        42.0
3        39.0
4        43.0
         ... 
9997     36.0
9998     42.0
9999     42.0
10000    28.0
10001    28.0
Name: Age, Length: 10002, dtype: float64

In [65]:
# # Fill the null value of IsActiveMember column
data['IsActiveMember'].value_counts()
data['IsActiveMember']=data['IsActiveMember'].fillna(1.0)


In [66]:
# Fill the null value of HasCrCard column
data['HasCrCard'].value_counts()
data['HasCrCard']=data['HasCrCard'].fillna(1.0)

In [67]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 10002 entries, 0 to 10001
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      10002 non-null  int64  
 1   Geography        10002 non-null  str    
 2   Gender           10002 non-null  int64  
 3   Age              10002 non-null  float64
 4   Tenure           10002 non-null  int64  
 5   Balance          10002 non-null  float64
 6   NumOfProducts    10002 non-null  int64  
 7   HasCrCard        10002 non-null  float64
 8   IsActiveMember   10002 non-null  float64
 9   EstimatedSalary  10002 non-null  float64
 10  Exited           10002 non-null  int64  
dtypes: float64(5), int64(5), str(1)
memory usage: 918.3 KB


In [68]:
data.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='str')

In [69]:
# Convert geography column into one-hot-encoding
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10002 stored elements and shape (10002, 3)>

In [70]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [0., 1., 0.],
       [1., 0., 0.],
       [1., 0., 0.]], shape=(10002, 3))

In [71]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [72]:
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42.0,2,0.00,1,1.0,1.0,101348.88,1
1,608,Spain,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0
2,502,France,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1
3,699,France,0,39.0,1,0.00,2,0.0,0.0,93826.63,0
4,850,Spain,0,43.0,2,125510.82,1,1.0,1.0,79084.10,0


In [73]:
geo_encoder_df=pd.DataFrame(geo_encoder.toarray(), columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoder_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0
9999,0.0,1.0,0.0
10000,1.0,0.0,0.0


In [74]:
# combine one hot encoder columns with the original dataframe and remove the existing geography column
data=pd.concat([data.drop('Geography', axis=1), geo_encoder_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42.0,2,0.00,1,1.0,1.0,101348.88,1,1.0,0.0,0.0
1,608,0,41.0,1,83807.86,1,0.0,1.0,112542.58,0,0.0,0.0,1.0
2,502,0,42.0,8,159660.80,3,1.0,0.0,113931.57,1,1.0,0.0,0.0
3,699,0,39.0,1,0.00,2,0.0,0.0,93826.63,0,1.0,0.0,0.0
4,850,0,43.0,2,125510.82,1,1.0,1.0,79084.10,0,0.0,0.0,1.0


In [75]:
# Save the label encoder and one hot encoder technique here
with open('label_encoder_gender.pkl', 'wb')as file:
    pickle.dump(label_encoder_gender, file)
    
with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

In [76]:
# Divide the dataset into independent and dependent feature (train and test data)
X= data.drop('Exited', axis=1)
y=data['Exited']

# split the data into training and testing sets
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2, random_state=42)

# Scaling the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)
X_train

array([[-0.19344335,  0.91360328,  0.29613055, ..., -0.99737876,
         1.72529422, -0.57710974],
       [-0.95045188,  0.91360328, -1.42278638, ..., -0.99737876,
        -0.57961128,  1.73277271],
       [ 0.10728607, -1.094567  , -0.84981407, ...,  1.00262813,
        -0.57961128, -0.57710974],
       ...,
       [ 0.87466458, -1.094567  , -0.08585099, ...,  1.00262813,
        -0.57961128, -0.57710974],
       [ 0.16950595,  0.91360328,  0.39162594, ...,  1.00262813,
        -0.57961128, -0.57710974],
       [ 0.48060534,  0.91360328,  1.15558902, ..., -0.99737876,
         1.72529422, -0.57710974]], shape=(8001, 12))

In [77]:
with open('scaler.pkl', 'wb')as file:
    pickle.dump(scaler, file)

### ANN IMPLEMENTATION

In [78]:
import tensorflow as tf
from tensorflow.keras.models import Sequential #type:ignore
from tensorflow.keras.layers import Dense      #type:ignore
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard  #type:ignore
import datetime

In [79]:
# Build our ANN model
model=Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # 1st hidden layer
    Dense(32, activation='relu'),# 2nd hidden layer
    Dense(1, activation='sigmoid') # output layer
])

e:\Generative AI\Deep learning\dlenv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [80]:
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [81]:
# compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [82]:
# set up the tensorboard (vizualization)
log_dir="logs/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir, histogram_freq=1)
tensorflow_callback

In [83]:
# Set up Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
early_stopping_callback

In [84]:
# Train the model
history=model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=100, callbacks=[early_stopping_callback, tensorflow_callback])

Epoch 1/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.8018 - loss: 0.4455 - val_accuracy: 0.8316 - val_loss: 0.3939
Epoch 2/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8416 - loss: 0.3799 - val_accuracy: 0.8606 - val_loss: 0.3587
Epoch 3/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8554 - loss: 0.3533 - val_accuracy: 0.8611 - val_loss: 0.3499
Epoch 4/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8595 - loss: 0.3445 - val_accuracy: 0.8581 - val_loss: 0.3494
Epoch 5/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8609 - loss: 0.3382 - val_accuracy: 0.8626 - val_loss: 0.3416
Epoch 6/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8601 - loss: 0.3349 - val_accuracy: 0.8631 - val_loss: 0.3429
Epoch 7/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8613 - loss: 0.3328 - val_accuracy: 0.8601 - val_loss: 0.3449
Epoch 8/100
251/251 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8634 - loss: 0.3297 - val_accu

In [85]:
model.save('model.h5')

In [129]:
# to do the vizualization we have to load the Tensorboard Extention
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [122]:
import os, sys
scripts = os.path.join(os.path.dirname(sys.executable), "Scripts")
print(scripts)
print(os.listdir(scripts)[:50])   # check that tensorboard.exe is listed
os.environ["TENSORBOARD_BINARY"] = os.path.join(scripts, "tensorboard.exe")

e:\Generative AI\Deep learning\dlenv\Scripts
['2to3-script.py', '2to3.exe', 'curl-cffi.exe', 'datasets-cli.exe', 'deep-translator.exe', 'dt.exe', 'f2py.exe', 'get_gprof', 'get_objgraph', 'hf.exe', 'httpx.exe', 'huggingface-cli.exe', 'idle-script.py', 'idle.exe', 'idna.exe', 'isympy.exe', 'jsonschema.exe', 'markdown-it.exe', 'markdown_py.exe', 'nltk.exe', 'normalizer.exe', 'numpy-config.exe', 'pip-script.py', 'pip.exe', 'pip3-script.py', 'pip3.exe', 'plotly_get_chrome.exe', 'pwiz.exe', 'pydoc-script.py', 'pydoc.exe', 'sacremoses.exe', 'sample.exe', 'saved_model_cli.exe', 'streamlit.exe', 'tensorboard.exe', 'tfds.exe', 'tflite_convert.exe', 'tf_upgrade_v2.exe', 'tiny-agents.exe', 'toco.exe', 'torchfrtrace.exe', 'torchrun.exe', 'tqdm.exe', 'transformers-cli.exe', 'typer.exe', 'undill', 'uvicorn.exe', 'watchmedo.exe', 'websockets.exe', 'wheel-script.py']


In [130]:
%reload_ext tensorboard
%tensorboard --logdir logs/fit --port 6013

Reusing TensorBoard on port 6013 (pid 20196), started 2:28:35 ago. (Use '!kill 20196' to kill it.)